<a href="https://colab.research.google.com/github/SrijanKumar123/flyrank-ml-internship/blob/main/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SrijanKumar123/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
%pip -q install duckdb
import duckdb
from google.colab import userdata

token = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute(f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{token}'
    )
""")

In [ ]:
REL = """
read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

In [ ]:
%pip -q install -U duckdb huggingface_hub

import duckdb
from google.colab import userdata
from huggingface_hub import whoami

token = userdata.get("HF_TOKEN")

print("Token found:", token is not None)
print("Logged in as:", whoami(token=token)["name"])

Token found: True
Logged in as: srijan317


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

Funnel Breakdown: Out of 9.84M evaluated records, 82.6K pages (0.8%) require immediate human review, successfully filtering out 99.2% of noise.

Primary Leverage Area: Snippet optimization (LOW_CTR_PAGE_ONE) represents the largest opportunity pool with 70,618 instances, followed by striking-distance content refreshes (6,780 instances).

Queue Order: Rules are evaluated hierarchically so high-intent Page 1 wins are prioritized over low-engagement layout fixes.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np
import pandas as pd

dataset = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        gsc_avg_position,
        gsc_impressions,
        gsc_clicks,
        ga4_pageviews,
        ga4_sessions,
        ga4_total_engagement_sec,
        scroll_events,
        sessions_ai
    FROM {REL}
""").df()

dataset["ctr"] = dataset["gsc_clicks"] / dataset["gsc_impressions"].replace(
    0, np.nan
).fillna(0)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
dataset["gsc_avg_position"] = dataset["gsc_avg_position"].fillna(100.0)
dataset["gsc_impressions"] = dataset["gsc_impressions"].fillna(0)
dataset["ga4_pageviews"] = dataset["ga4_pageviews"].fillna(0)
dataset["scroll_events"] = dataset["scroll_events"].fillna(0)
dataset["sessions_ai"] = dataset["sessions_ai"].fillna(0)
dataset["ctr"] = dataset["ctr"].fillna(0)

#Creates boolean masks
cond_low_ctr = (
    (dataset["gsc_avg_position"] <= 10) &
    (dataset["gsc_impressions"] >= 500) &
    (dataset["ctr"] < 0.02)
).fillna(False)

cond_striking = (
    (dataset["gsc_avg_position"] > 3) &
    (dataset["gsc_avg_position"] <= 20) &
    (dataset["gsc_impressions"] >= 500)
).fillna(False)

cond_low_eng = (
    (dataset["ga4_pageviews"] > 100) &
    (dataset["scroll_events"] == 0)
).fillna(False)

cond_ai = (dataset["sessions_ai"] > 0).fillna(False)

conditions = [cond_low_ctr, cond_striking, cond_low_eng, cond_ai]
choices = [
    "LOW_CTR_PAGE_ONE",
    "STRIKING_DISTANCE_BOOST",
    "LOW_ENGAGEMENT_FIX",
    "PROTECT_AI_WINNER"
]

dataset["reason_code"] = np.select(conditions, choices, default="MONITOR_ONLY")

print(dataset["reason_code"].value_counts())

reason_code
MONITOR_ONLY               9758777
LOW_CTR_PAGE_ONE             70618
STRIKING_DISTANCE_BOOST       6780
PROTECT_AI_WINNER             5199
LOW_ENGAGEMENT_FIX               4
Name: count, dtype: int64


In [ ]:
print(con.sql(f"SELECT * FROM {REL} LIMIT 0").df().columns.tolist())

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Audience & Primary Use Case
* **Primary Users:** Content strategists, SEO managers, and editorial leaders.
* **Core Function:** Functions as a **decision-support triage tool** that automatically parses raw performance data across millions of URLs to identify high-leverage optimization opportunities. It replaces manual spreadsheet filtering with standardized, rule-based prioritization.

### Triage Performance & Coverage
* **Noise Filtering Efficiency:** Out of **9,841,378 total performance records**, the playbook successfully filters out **99.16% (9,758,777 records)** as routine background noise, isolating a tight, actionable queue of **82,601 high-signal opportunities (0.84%)**.
* **Portfolio Breadth:** Generates active recommendations across **38 out of 55 client portfolios (69.1%)**, ensuring broad operational reach while remaining completely silent on healthy or inactive domains.

### System Boundaries & Known Limits
1. **Unindexed / Zero-Traffic Cold Start:** Exactly **100% of records with under 500 impressions (9,735,612 records)** default to `MONITOR_ONLY`. The system relies on search demand signals and cannot evaluate brand-new or unindexed URLs.
2. **Fixed Impression Cutoffs:** The $\ge 500$ impression threshold is tuned for broad portfolio search volumes, but may filter out low-volume, high-converting enterprise B2B pages.
3. **External Macro Factors:** The playbook evaluates historical daily metrics and cannot account for real-time technical outages, major Google core updates, or seasonal search demand swings.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
total_records = len(dataset)
actionable_records = (dataset["reason_code"] != "MONITOR_ONLY").sum()
actionable_pct = (actionable_records / total_records) * 100

print("=== PORTFOLIO TRIAGE SUMMARY ===")
print(f"Total Portfolio Records: {total_records:,}")
print(f"Actionable Queue Items:  {actionable_records:,} ({actionable_pct:.2f}%)")
print(f"Filtered Out Noise:      {total_records - actionable_records:,} ({100 - actionable_pct:.2f}%)\n")

client_summary = dataset.groupby("client_hash_id")["reason_code"].apply(
    lambda x: (x != "MONITOR_ONLY").sum()
).reset_index(name="actionable_count")

total_clients = dataset["client_hash_id"].nunique()
clients_with_actions = (client_summary["actionable_count"] > 0).sum()

print("=== CLIENT COVERAGE ===")
print(f"Total Unique Clients: {total_clients}")
print(f"Clients with Actionable Items: {clients_with_actions} ({clients_with_actions / total_clients * 100:.1f}%)\n")

low_imp_count = (dataset["gsc_impressions"] < 500).sum()
low_imp_monitor = ((dataset["gsc_impressions"] < 500) & (dataset["reason_code"] == "MONITOR_ONLY")).sum()

print("=== SYSTEM LIMITATION PROOF ===")
print(f"Records with < 500 Impressions: {low_imp_count:,}")
print(f"Low-Impression Defaulted to MONITOR_ONLY: {low_imp_monitor:,} ({low_imp_monitor / low_imp_count * 100:.1f}%)")

=== PORTFOLIO TRIAGE SUMMARY ===
Total Portfolio Records: 9,841,378
Actionable Queue Items:  82,601 (0.84%)
Filtered Out Noise:      9,758,777 (99.16%)

=== CLIENT COVERAGE ===
Total Unique Clients: 55
Clients with Actionable Items: 38 (69.1%)

=== SYSTEM LIMITATION PROOF ===
Records with < 500 Impressions: 9,739,927
Low-Impression Defaulted to MONITOR_ONLY: 9,735,612 (100.0%)


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Pre-Action Human Verification Protocol
Before executing any recommendation generated by the queue, human strategists must verify three core criteria:
1. **Search Intent Alignment:** For `LOW_CTR_PAGE_ONE` tasks, verify that proposed metadata updates directly reflect search query intent before modifying Page 1 title tags.
2. **CMS Recency Check:** Confirm in the CMS that the target URL was not refreshed within the last 30–60 days to prevent redundant or conflicting updates.
3. **Technical Health Audit:** Verify that non-performing pages are not suffering from broken layouts, 404 errors, or improper canonical tags prior to rewriting copy.

### Risk-Tiering Framework
Actionable queue items are categorized into two distinct human review tiers based on current traffic exposure and revenue risk:
* **`STANDARD_REVIEW` (81,989 items / 99.26% of queue):** Standard editorial review for striking-distance updates and layout fixes where optimization upside far outweighs existing risk.
* **`SENIOR_EDITORIAL_SIGN_OFF` (612 items / 0.74% of queue):** Senior strategist approval required prior to editing high-stakes assets—specifically **599 high-impression Page 1 pages**, **9 major AI referral drivers**, and **4 high-traffic striking-distance pages**. Modifying these assets directly impacts established top-of-funnel traffic.

### The "No-Go" List (Strictly Prohibited from Automation)
1. **NO Direct-to-Live Automated Publishing:** LLM-generated text or metadata must never publish directly to live sites without human editorial approval.
2. **NO Automated URL or Slug Changes:** Changing URL structures or deleting content without manual 301-redirect mapping creates critical technical debt and broken links.
3. **NO Automated Edits to Core Business Pages:** Terms of service, pricing tables, legal disclaimers, and conversion landing pages are strictly excluded from automated optimization queues.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

filtered = dataset[dataset["reason_code"] != "MONITOR_ONLY"]

high_risk_condition = (
    (filtered["gsc_avg_position"] <= 10) & (filtered["gsc_impressions"] >= 5000)
) | (filtered["sessions_ai"] >= 10)

filtered["review_level"] = np.where(
    high_risk_condition,
    "SENIOR_EDITORIAL_SIGN_OFF",
    "STANDARD_REVIEW"
)

print("=== HUMAN REVIEW RISK TIER BREAKDOWN ===")
print(filtered["review_level"].value_counts())
print("\n=== HIGH-RISK ACTIONS NEEDING SENIOR SIGN-OFF ===")
print(filtered[filtered["review_level"] == "SENIOR_EDITORIAL_SIGN_OFF"]["reason_code"].value_counts())

=== HUMAN REVIEW RISK TIER BREAKDOWN ===
review_level
STANDARD_REVIEW              81989
SENIOR_EDITORIAL_SIGN_OFF      612
Name: count, dtype: int64

=== HIGH-RISK ACTIONS NEEDING SENIOR SIGN-OFF ===
reason_code
LOW_CTR_PAGE_ONE           599
PROTECT_AI_WINNER            9
STRIKING_DISTANCE_BOOST      4
Name: count, dtype: int64


/tmp/ipykernel_836/1190544920.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered["review_level"] = np.where(


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Decay & Refresh Cadence
* **Why Recommendations Decay:** Search engine results pages (SERPs) are dynamic. Position ranks, CTRs, and impression volumes fluctuate constantly due to competitor updates, seasonality, and Google core algorithm updates.
* **Rolling Refresh Schedule:** Action queue outputs should be regenerated on a **weekly or bi-weekly cadence** to ensure editorial teams act on live, accurate performance signals rather than stale historical snapshots.

### System Monitoring & Retrain Triggers
1. **Distribution Shift / Data Pipeline Drift:**
   * **Trigger Boundary:** Healthy baseline ratio is set between **0.1% and 5.0%** of total records.
   * **Alert Logic:** If actionable records drop below 0.1% (indicating dropped GSC/GA4 records) or spike above 5.0% (indicating loose rules or a major search update), trigger an immediate pipeline and threshold audit.
2. **Post-Optimization Performance Decay:**
   * **Trigger:** If URLs updated under `LOW_CTR_PAGE_ONE` or `STRIKING_DISTANCE_BOOST` fail to show measurable impression or position lift within **60 to 90 days** post-refresh.
   * **Action:** Re-evaluate and re-calibrate position and impression cutoff parameters (`gsc_avg_position`, `gsc_impressions`).
3. **Model & Feature Drift:**
   * **Trigger:** If overall portfolio CTR drops across multiple domains, re-tune threshold values to reflect new SERP layout realities (e.g., increased AI Overview presence compressing standard organic CTR).

### Cost / Value Workload Framework
* **Full Queue Workload:** Processing the entire actionable queue requires **~9,251 human reviewer hours** (~5,861 hours for 70,618 snippet fixes + ~3,390 hours for 6,780 content refreshes).
* **High-ROI Quick Wins Strategy:** By sorting `LOW_CTR_PAGE_ONE` assets by raw impression volume and focusing on the **Top 100 highest-impression quick wins**, editorial teams can capture maximum click lift in just **8.30 human reviewer hours**, dramatically improving ROI per strategist hour worked.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

total_recs = len(dataset)
filtered_cnt = (dataset["reason_code"] != "MONITOR_ONLY").sum()
filtered_ratio = filtered_cnt / total_recs

def baseline_health(filtered_ratio):
  lower_threshold = 0.1
  upper_threshold = 5.0

  if filtered_ratio < lower_threshold:
    print("Possible data pipeline error")

  elif filtered_ratio > upper_threshold:
    print("Rules are too lose")

  else:
    print("Clean health check")

  baseline_health(filtered_ratio * 100)

In [ ]:
snippet_time = 0.083
refresh_time = 0.5

low_ctr_count = (dataset["reason_code"] == "LOW_CTR_PAGE_ONE").sum()
striking_distance_count = (dataset["reason_code"] == "STRIKING_DISTANCE_BOOST").sum()

snippet_hours = low_ctr_count * snippet_time
refresh_hours = striking_distance_count * refresh_time

total_hours = snippet_hours + refresh_hours

print(f"\nTotal Estimated Workload: {total_hours:.2f} hours")


Total Estimated Workload: 9251.29 hours


In [ ]:
HOURS_PER_SNIPPET_FIX = 0.083

filtered_df = dataset[dataset["reason_code"] == "LOW_CTR_PAGE_ONE"].sort_values(by="gsc_impressions", ascending=False)
filtered_count = len(filtered_df)

top_100 = filtered_df.head(100)
top_100_count = len(top_100)

total_queue_hours = filtered_count * HOURS_PER_SNIPPET_FIX
quick_win_hours = top_100_count * HOURS_PER_SNIPPET_FIX

print(f"Total queue: {filtered_count:,} items")
print(f"Total queue workload: {total_queue_hours:.2f} hours")

print(f"\nTop 100 quick wins: {top_100_count:,} items")
print(f"Quick-win workload: {quick_win_hours:.2f} hours")

Total queue: 70,618 items
Total queue workload: 5861.29 hours

Top 100 quick wins: 100 items
Quick-win workload: 8.30 hours


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import json

os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

actionable_queue = dataset[dataset["reason_code"] != "MONITOR_ONLY"].copy()

actionable_queue = actionable_queue.sort_values(by="gsc_impressions", ascending=False)

export_cols = [
    "client_hash_id",
    "content_hash_id",
    "reason_code",
    "gsc_avg_position",
    "gsc_impressions",
    "gsc_clicks",
    "ctr",
    "sessions_ai",
    "ga4_pageviews"
]

if "review_level" in dataset.columns:
    export_cols.append("review_level")

queue_csv_path = "work/outputs/ranked_action_queue.csv"
actionable_queue[export_cols].to_csv(queue_csv_path, index=False)
print(f"✅ Exported ranked action queue ({len(actionable_queue):,} items) to: {queue_csv_path}")

metrics_data = {
    "total_portfolio_records": int(len(dataset)),
    "actionable_queue_items": int(len(actionable_queue)),
    "filtered_noise_items": int((dataset["reason_code"] == "MONITOR_ONLY").sum()),
    "actionable_ratio_pct": round(float(len(actionable_queue) / len(dataset) * 100), 4),
    "total_unique_clients": int(dataset["client_hash_id"].nunique()),
    "clients_with_actionable_items": int(actionable_queue["client_hash_id"].nunique()),
    "reason_code_counts": dataset["reason_code"].value_counts().to_dict(),
    "estimated_total_workload_hours": round(float(
        (dataset["reason_code"] == "LOW_CTR_PAGE_ONE").sum() * 0.083 +
        (dataset["reason_code"] == "STRIKING_DISTANCE_BOOST").sum() * 0.5
    ), 2)
}

metrics_json_path = "work/outputs/playbook_metrics.json"
with open(metrics_json_path, "w") as f:
    json.dump(metrics_data, f, indent=2)

print(f"✅ Exported summary metrics receipt to: {metrics_json_path}")

✅ Exported ranked action queue (82,601 items) to: work/outputs/ranked_action_queue.csv
✅ Exported summary metrics receipt to: work/outputs/playbook_metrics.json


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.